In [1]:
"""
exp01 Step 03: Walk-Forward Evaluation (PROTOCOL.md §9-§12)

Per fold (3-month windows, 10-session embargo):
    1. VALIDATION: for each training window L in {3, 5, 10} years, fit one LightGBM P(TP) model per delta on the sampled
       training rows, decide at EVERY eligible minute of the validation quarter (best expected net return across deltas,
       buy above a threshold), and simulate each (L, threshold) candidate.
    2. SELECTION: validation total return of the quarter (ties: shorter window, higher threshold).
    3. HONEST REPORTING (every run mode, validation data only):
         - every candidate chained over the validation quarters (after the fact: optimistic for the best one);
         - the PRIOR-ONLY path: the setting selected at fold f - 1, on fold f's validation quarter (its decisions of that
           quarter), together with RANDOM_BASELINE_RUN_COUNT random-entry runs of matched activity and the fixed-time entry;
         - time invested (exposure) and the zero-skill reference: a strategy without skill that is invested a fraction f of
           the time is expected to grow about (1 + buy-and-hold)^f - 1;
         - the minimum detectable effect of the quarterly excess returns.
       exp01 positions can extend past the end of a quarter (up to 10 sessions), so the path is summarized per quarter
       (no daily chaining or daily bootstrap here).
    4. TEST (run modes "latest" / "history" only): refit and evaluate the test window ONCE, with the same baselines.

RUN_MODE: "validation_only" (use until the design is frozen), "latest", "history".
Model and decision policy: one LightGBM classifier per delta + expected-return rule (walk_forward.py), a proposal that was
never formally agreed (PROTOCOL.md §10).
Prerequisite: exp01 step 01. Outputs: store04_experiments/exp01_minute_entry/step03_walk_forward_data/.
Run time: about 1 minute per fold for the validation candidates, plus the prior-only baselines (about 45 folds).
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT DATA LOADING AND EXECUTION FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import read_model_dataset_pdf, get_model_feature_col_str_list
from experiments.exp01_minute_entry.walk_forward import get_fold_pdf, run_walk_forward_fold_dict, run_window_with_baseline_dict
# IMPORT THE EVALUATION FUNCTIONS
from so.core.evaluation import get_pooled_selection_pdf, get_prior_only_pdf, get_candidate_summary_pdf, get_window_return_summary_dict, get_baseline_summary_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning
# IMPORT PLOT FUNCTIONS
from so.features.plot_ohlcv_utils import plot_transaction_ohlcv

In [3]:
# DEFINE THE RUN MODE ("validation_only", "latest" or "history"): NEVER "latest" / "history" BEFORE THE DESIGN IS FROZEN
RUN_MODE_STR = "validation_only"
# DEFINE WHETHER THE TEST WINDOWS ARE EVALUATED
RUN_TEST_BOOL = RUN_MODE_STR in ["latest", "history"]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tTest Windows Evaluated:\t{RUN_TEST_BOOL}")
print(f"Deltas:\t{exp_config.MODEL_DELTA_LIST}")
print(f"Training Windows:\t{exp_config.TRAIN_WINDOW_YEARS_LIST} years\tThresholds:\t{exp_config.EV_THRESHOLD_LIST}")
print(f"Sampling:\t{exp_config.SAMPLING_MODE} ({exp_config.SAMPLE_EVERY_N_MINUTES} min)\tWeights:\t{exp_config.SAMPLE_WEIGHT_MODE}\tEmbargo:\t{exp_config.EMBARGO_TRADING_DAYS} sessions")

Run Mode:	validation_only	Test Windows Evaluated:	False
Deltas:	[0.001, 0.0011, 0.0013, 0.0014, 0.0016, 0.0018, 0.002, 0.0022, 0.0025, 0.0028, 0.0032, 0.0035, 0.004, 0.0045, 0.005, 0.0056, 0.0063, 0.0071, 0.0079, 0.0089, 0.01]
Training Windows:	[3, 5, 10] years	Thresholds:	[0.0, 0.0002, 0.0005, 0.001]
Sampling:	interval (15 min)	Weights:	uniqueness	Embargo:	10 sessions


In [4]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step03_walk_forward_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

Experiment:	exp01_minute_entry	Protocol:	1.0	Config Hash:	16493e598a
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step03_walk_forward_data/
ℹ️ No earlier 'validation' entry of exp01_minute_entry in the trial log (config hash 16493e598a).


In [5]:
# CALL FUNCTION TO READ THE COMPLETE MINUTE BARS (RAW FOLDER OF so.paths)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS (USED BY THE SIMULATOR AND THE SAMPLE WEIGHTS)
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# CALL FUNCTION TO GET THE WALK-FORWARD FOLDS
fold_pdf = get_fold_pdf(ohlcv_array_dict["session_date_list"])
# SELECT THE FOLDS OF THE RUN MODE
selected_fold_pdf = fold_pdf.iloc[[-1]] if RUN_MODE_STR == "latest" else fold_pdf
# SAVE THE SCHEDULE
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}\tmax timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
print(f"Fold Count:\t{len(fold_pdf)}\tSelected:\t{len(selected_fold_pdf)}")
# PREVIEW DATAFRAME
selected_fold_pdf

min timestamp:	2005-01-03 09:30:00-05:00	max timestamp:	2026-08-13 15:59:00-04:00
Fold Count:	45	Selected:	45


,fold_id,test_start,test_end,valid_start,valid_end,train_end,refit_end,train_start_3y,refit_start_3y,fits_3y,train_start_5y,refit_start_5y,fits_5y,train_start_10y,refit_start_10y,fits_10y
0,0,2015-05-14,2015-08-13,2015-01-30,2015-04-29,2015-01-14,2015-04-29,2012-01-17,2012-04-30,True,2010-01-15,2010-04-30,True,2005-01-18,2005-05-02,True
1,1,2015-08-14,2015-11-13,2015-05-01,2015-07-30,2015-04-16,2015-07-30,2012-04-17,2012-07-31,True,2010-04-19,2010-08-02,True,2005-04-18,2005-08-01,True
2,2,2015-11-16,2016-02-12,2015-07-31,2015-10-30,2015-07-16,2015-10-30,2012-07-17,2012-10-31,True,2010-07-19,2010-11-01,True,2005-07-18,2005-10-31,True
3,3,2016-02-16,2016-05-13,2015-10-30,2016-01-29,2015-10-15,2016-01-29,2012-10-16,2013-01-30,True,2010-10-18,2011-01-31,True,2005-10-17,2006-01-30,True
4,4,2016-05-16,2016-08-12,2016-02-01,2016-04-29,2016-01-14,2016-04-29,2013-01-15,2013-04-30,True,2011-01-18,2011-05-02,True,2006-01-17,2006-05-01,True
5,5,2016-08-15,2016-11-11,2016-05-02,2016-07-29,2016-04-15,2016-07-29,2013-04-16,2013-07-30,True,2011-04-18,2011-08-01,True,2006-04-17,2006-07-31,True
6,6,2016-11-14,2017-02-13,2016-07-29,2016-10-28,2016-07-14,2016-10-28,2013-07-15,2013-10-29,True,2011-07-15,2011-10-31,True,2006-07-17,2006-10-30,True
7,7,2017-02-14,2017-05-12,2016-10-31,2017-01-30,2016-10-14,2017-01-30,2013-10-15,2014-01-31,True,2011-10-17,2012-01-31,True,2006-10-16,2007-01-31,True
8,8,2017-05-15,2017-08-11,2017-01-30,2017-04-28,2017-01-12,2017-04-28,2014-01-13,2014-04-29,True,2012-01-13,2012-04-30,True,2007-01-16,2007-04-30,True
9,9,2017-08-14,2017-11-13,2017-05-01,2017-07-28,2017-04-13,2017-07-28,2014-04-14,2014-07-29,True,2012-04-16,2012-07-30,True,2007-04-16,2007-07-30,True


In [6]:
# COLLECT THE EARLIEST TRAINING START AND THE LATEST REFIT END OF THE SELECTED FOLDS
pool_start_date = min(selected_fold_pdf[f"train_start_{train_years}y"].min() for train_years in exp_config.TRAIN_WINDOW_YEARS_LIST)
pool_end_date = selected_fold_pdf["refit_end"].max()
# CALL FUNCTION TO READ THE SAMPLED TRAINING POOL (EVERY TRAINING / REFIT WINDOW IS A SLICE OF IT)
train_pool_pdf = read_model_dataset_pdf(str(pool_start_date), str(pool_end_date), exp_config.MODEL_DELTA_LIST, exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES)
# DEFINE THE MODEL FEATURES
feature_col_str_list = get_model_feature_col_str_list(train_pool_pdf)
# DISPLAY INFORMATION
print(f"Training Pool:\t{pool_start_date} -> {pool_end_date}\t{len(train_pool_pdf):,} rows")
print(f"Model Features ({len(feature_col_str_list)}):\t{feature_col_str_list}")

Reading 5,353 model dataset files (2005-01-18 -> 2026-04-29, sampling: 'interval')
Training Pool:	2005-01-18 -> 2026-04-29	127,924 rows
Model Features (33):	['trend', 'segment', 'color', 'bot_wick_pct', 'body_pct', 'top_wick_pct', 'stoch_k', 'stoch_d', 'rsi', 'bb_pct', 'dc_pct', 'ms_minima_trend', 'ms_maxima_trend', 'ms_low_status', 'ms_high_status', 'ms_trend', 'minima_count', 'maxima_count', 'prev_seg_delta_pct', 'seg_cs_count', 'seg', 'minutes_since_open', 'minutes_to_close', 'day_of_week', 'overnight_gap_pct', 'prev_day_return_pct', 'prev_day_range_pct', 'prev_day_high_dist_pct', 'prev_day_low_dist_pct', 'intraday_return_pct', 'daily_volatility', 'relative_volume', 'ath_drawdown_pct']


In [7]:
# LISTS TO HOLD THE CANDIDATES, THE PRIOR-ONLY PATH AND THE TEST SUMMARIES
candidate_pdf_list, prior_row_dict_list, prior_baseline_pdf_list, fold_summary_dict_list = [], [], [], []
# DEFINE THE SETTING SELECTED AT THE PREVIOUS FOLD (NONE BEFORE THE FIRST FOLD)
previous_best_setting_dict = None
# ITERATE OVER THE SELECTED FOLDS
for _, fold_row in selected_fold_pdf.iterrows():
    # DEFINE THE FOLD ID AND START TIME
    fold_id, start_time = int(fold_row["fold_id"]), time.time()
    # DISPLAY INFORMATION
    print(f"\nFold {fold_id}:\tvalidation {fold_row['valid_start']} -> {fold_row['valid_end']}\ttest {fold_row['test_start']} -> {fold_row['test_end']}")
    # CALL FUNCTION TO READ THE UNSAMPLED VALIDATION AND TEST ROWS (EVERY ELIGIBLE DECISION, AS A LIVE MODEL)
    eval_pool_pdf = read_model_dataset_pdf(str(fold_row["valid_start"]), str(fold_row["test_end"]), exp_config.MODEL_DELTA_LIST, "all", alert_in=False)
    # CALL FUNCTION TO RUN THE FOLD
    fold_result_dict = run_walk_forward_fold_dict(fold_row, train_pool_pdf, eval_pool_pdf, ohlcv_array_dict, feature_col_str_list, run_test_bool_in=RUN_TEST_BOOL)
    fold_candidate_pdf = fold_result_dict["selection_pdf"]
    candidate_pdf_list.append(fold_candidate_pdf)
    # LOG EVERY VALIDATION CANDIDATE (ONE WRITE PER FOLD) AND SAVE THEM
    log_trial_pdf(exp_config, "validation", fold_candidate_pdf, ["train_years", "ev_threshold"], [col for col in fold_candidate_pdf.columns if col.startswith("valid_")], False, fold_pdf)
    write_csv_file_to_path(fold_candidate_pdf, f"{output_path_str}fold{fold_id:03d}_validation_candidate_data.csv", "W", alert_in=False)
    # PRIOR-ONLY PATH: THE SETTING SELECTED AT THE PREVIOUS FOLD, ON THIS FOLD'S VALIDATION QUARTER, WITH ITS BASELINES
    if previous_best_setting_dict is not None:
        decision_pdf = fold_result_dict["valid_decision_pdf_dict"][(previous_best_setting_dict["train_years"], previous_best_setting_dict["ev_threshold"])]
        window_dict = run_window_with_baseline_dict(decision_pdf, fold_result_dict["valid_pdf"], ohlcv_array_dict, fold_row["valid_start"], fold_row["valid_end"])
        prior_row_dict_list.append({"fold_id": fold_id, "selected_at_fold_id": fold_id - 1, **previous_best_setting_dict,
                                    **{f"valid_{key}": value for key, value in window_dict["metric_dict"].items()},
                                    "valid_buy_hold_total_return": window_dict["metric_dict"]["buy_hold_return"]})
        prior_baseline_pdf_list.append(window_dict["baseline_pdf"].assign(fold_id=fold_id))
        print(f"\tPrior-only ({previous_best_setting_dict}):\t{window_dict['metric_dict']['total_return']:.2%} vs buy & hold {window_dict['metric_dict']['buy_hold_return']:.2%}")
    # REMEMBER THIS FOLD'S SELECTION FOR THE NEXT FOLD
    previous_best_setting_dict = fold_result_dict["best_setting_dict"]
    # IF THE TEST WINDOW WAS EVALUATED
    if RUN_TEST_BOOL:
        # LOG THE TEST EVALUATION
        log_trial_dict(exp_config, "test", fold_result_dict["best_setting_dict"], fold_result_dict["test_metric_dict"], True, fold_row)
        # SAVE THE TEST OUTPUTS
        write_csv_file_to_path(fold_result_dict["baseline_metric_pdf"], f"{output_path_str}fold{fold_id:03d}_test_baseline_data.csv", "W", alert_in=False)
        write_csv_file_to_path(fold_result_dict["cost_sensitivity_pdf"], f"{output_path_str}fold{fold_id:03d}_test_cost_sensitivity_data.csv", "W", alert_in=False)
        write_csv_file_to_path(fold_result_dict["test_transaction_pdf"], f"{output_path_str}fold{fold_id:03d}_test_transaction_data.csv", "W", alert_in=False)
        write_csv_file_to_path(fold_result_dict["test_daily_equity_pdf"], f"{output_path_str}fold{fold_id:03d}_test_daily_equity_data.csv", "W", alert_in=False)
        # COLLECT THE RANDOM BASELINE RETURNS AND DEFINE THE FOLD SUMMARY
        baseline_pdf = fold_result_dict["baseline_metric_pdf"]
        random_return_arr = baseline_pdf[baseline_pdf["baseline"].str.startswith("random_")]["total_return"].to_numpy()
        fold_summary_dict_list.append({"fold_id": fold_id, "test_start": fold_row["test_start"], "test_end": fold_row["test_end"],
                                       **fold_result_dict["best_setting_dict"], **fold_result_dict["test_metric_dict"],
                                       "random_mean_return": random_return_arr.mean() if len(random_return_arr) else np.nan,
                                       "model_vs_random_percentile": (random_return_arr < fold_result_dict["test_metric_dict"]["total_return"]).mean() if len(random_return_arr) else np.nan,
                                       "fixed_time_return": baseline_pdf.loc[baseline_pdf["baseline"] == "fixed_time", "total_return"].iloc[0]})
    # DISPLAY INFORMATION
    print(f"\t⏱ Fold {fold_id} processed in {(time.time() - start_time) / 60:.1f} minutes")
# CONCATENATE THE CANDIDATES AND SAVE THEM
candidate_pdf = pd.concat(candidate_pdf_list, ignore_index=True)
write_csv_file_to_path(candidate_pdf, f"{output_path_str}validation_candidate_data.csv", "W")


Fold 0:	validation 2015-01-30 -> 2015-04-29	test 2015-05-14 -> 2015-08-13
	Train 3y: 2012-01-17 -> 2015-01-14 (17,987 rows)
	Train 5y: 2010-01-15 -> 2015-01-14 (30,059 rows)
	Train 10y: 2005-01-18 -> 2015-01-14 (60,088 rows)
	🏆 Selected on validation: {'train_years': 5, 'ev_threshold': 0.001} (total_return: 5.2549%)
	⏱ Fold 0 processed in 0.7 minutes

Fold 1:	validation 2015-05-01 -> 2015-07-30	test 2015-08-14 -> 2015-11-13
	Train 3y: 2012-04-17 -> 2015-04-16 (17,987 rows)
	Train 5y: 2010-04-19 -> 2015-04-16 (30,059 rows)
	Train 10y: 2005-04-18 -> 2015-04-16 (60,112 rows)
	🏆 Selected on validation: {'train_years': 3, 'ev_threshold': 0.0002} (total_return: -0.8188%)
	Prior-only ({'train_years': 5, 'ev_threshold': 0.001}):	-2.84% vs buy & hold 0.10%
	⏱ Fold 1 processed in 0.9 minutes

Fold 2:	validation 2015-07-31 -> 2015-10-30	test 2015-11-16 -> 2016-02-12
	Train 3y: 2012-07-17 -> 2015-07-16 (17,999 rows)
	Train 5y: 2010-07-19 -> 2015-07-16 (30,059 rows)
	Train 10y: 2005-07-18 -> 2015-

In [8]:
"""
After-the-fact view (OPTIMISTIC for the best setting: it is the best of 12 chosen with hindsight). valid_exposure_pct is
the share of the window's bars with an open position. Buy-and-hold is measured per candidate over the window start ->
later of the window end and its last exit, so chained_buy_hold_return differs slightly per candidate.
"""
# CALL FUNCTION TO SUMMARIZE EVERY CANDIDATE OVER THE VALIDATION QUARTERS
candidate_summary_pdf = get_candidate_summary_pdf(candidate_pdf, ["train_years", "ev_threshold"], ["valid_exposure_pct", "valid_trade_count", "valid_skipped_decision_count"])
# ADD THE ZERO-SKILL REFERENCE: (1 + BUY-AND-HOLD) ^ EXPOSURE - 1 (A STRATEGY WITHOUT SKILL, INVESTED THAT SHARE OF THE TIME)
candidate_summary_pdf["zero_skill_reference_return"] = (1 + candidate_summary_pdf["chained_buy_hold_return"]) ** candidate_summary_pdf["mean_valid_exposure_pct"] - 1
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# PREVIEW DATAFRAME
candidate_summary_pdf.round(4)

,train_years,ev_threshold,window_count,chained_return,chained_buy_hold_return,chained_excess_return,windows_beating_buy_hold,sign_test_p_value,mean_excess_return,std_excess_return,t_stat,mde_per_window,mde_per_year,mean_valid_exposure_pct,mean_valid_trade_count,mean_valid_skipped_decision_count,zero_skill_reference_return
0,3,0.0005,45,1.5808,2.3036,-0.7228,14,0.9967,-0.0056,0.0260,-1.4388,0.0109,0.0434,0.9043,76.8667,15526.3333,1.9466
1,10,0.0002,45,1.5304,2.4849,-0.9545,16,0.9822,-0.0069,0.0244,-1.9019,0.0102,0.0408,0.9224,84.8444,16051.1111,2.1631
2,5,0.0010,45,1.5008,2.3139,-0.8131,16,0.9822,-0.0054,0.0305,-1.1994,0.0127,0.0509,0.7957,60.3111,11613.1333,1.5944
3,10,0.0000,45,1.3491,2.4219,-1.0729,11,0.9999,-0.0081,0.0230,-2.3707,0.0096,0.0385,0.9469,110.5111,18578.3111,2.2057
4,10,0.0005,45,1.3406,2.3942,-1.0536,10,1.0000,-0.0086,0.0295,-1.9447,0.0123,0.0493,0.8779,69.0000,12940.0889,1.9238
5,3,0.0010,45,1.2806,2.0838,-0.8032,14,0.9967,-0.0068,0.0310,-1.4767,0.0129,0.0517,0.8419,63.1778,12911.9333,1.5809
6,3,0.0002,45,1.0132,2.1780,-1.1648,11,0.9999,-0.0102,0.0219,-3.1258,0.0091,0.0365,0.9321,93.1111,17551.6444,1.9380
7,5,0.0005,45,0.9982,2.3127,-1.3145,13,0.9988,-0.0104,0.0280,-2.4983,0.0117,0.0468,0.8783,74.3333,14690.6000,1.8633
8,5,0.0002,45,0.8565,2.3582,-1.5017,11,0.9999,-0.0129,0.0218,-3.9513,0.0091,0.0364,0.9150,93.4000,16906.2000,2.0296
9,10,0.0010,45,0.6629,2.1044,-1.4415,12,0.9996,-0.0143,0.0314,-3.0660,0.0131,0.0524,0.7618,54.6222,8759.9778,1.3703


In [9]:
"""
Prior-only path (the honest validation estimate) and the information test: the setting selected at fold f - 1 on fold f's
validation quarter, against random entries of matched activity and the fixed-time entry. No test window is read.
"""
# IF THE PRIOR-ONLY PATH HAS ROWS (NOT IN "latest" MODE)
if prior_row_dict_list:
    # CONVERT THE PATH AND THE BASELINES AND SAVE THEM
    prior_only_pdf = pd.DataFrame(prior_row_dict_list)
    prior_baseline_pdf = pd.concat(prior_baseline_pdf_list, ignore_index=True)
    write_csv_file_to_path(prior_only_pdf, f"{output_path_str}prior_only_data.csv", "W", alert_in=False)
    write_csv_file_to_path(prior_baseline_pdf, f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
    # CROSS-CHECK THE PATH WITH THE GENERIC SELECTION (SINGLE QUARTER, TIES: SHORTER WINDOW, HIGHER THRESHOLD)
    generic_prior_pdf = get_prior_only_pdf(candidate_pdf, get_pooled_selection_pdf(candidate_pdf, ["train_years", "ev_threshold"], "valid_total_return",
                                                                                   exp_config.PRIOR_SELECTION_QUARTER_COUNT, [("train_years", True), ("ev_threshold", False)]),
                                           ["train_years", "ev_threshold"])
    assert generic_prior_pdf[["fold_id", "train_years", "ev_threshold"]].astype(float).equals(prior_only_pdf[["fold_id", "train_years", "ev_threshold"]].astype(float))
    # SUMMARIZE THE PATH (QUARTERS) AND THE BASELINES (INFORMATION TEST)
    prior_summary_dict = get_window_return_summary_dict(prior_only_pdf["valid_total_return"].to_numpy(), prior_only_pdf["valid_buy_hold_total_return"].to_numpy())
    baseline_summary_pdf = get_baseline_summary_pdf(prior_baseline_pdf)
    mean_exposure = float(prior_only_pdf["valid_exposure_pct"].mean())
    zero_skill_reference = (1 + prior_summary_dict["chained_buy_hold_return"]) ** mean_exposure - 1
    # DISPLAY INFORMATION
    print(f"Prior-only path ({prior_summary_dict['window_count']} quarters):\t{prior_summary_dict['chained_return']:.2%} vs buy & hold {prior_summary_dict['chained_buy_hold_return']:.2%}"
          f"\t(beats buy & hold in {prior_summary_dict['windows_beating_buy_hold']} quarters, one-sided sign test p = {prior_summary_dict['sign_test_p_value']:.3f})")
    print(f"Mean excess per quarter:\t{prior_summary_dict['mean_excess_return']:.2%} (SD {prior_summary_dict['std_excess_return']:.2%}, t = {prior_summary_dict['t_stat']:.2f})"
          f"\tMinimum detectable effect:\t{prior_summary_dict['mde_per_window']:.2%} per quarter, about {prior_summary_dict['mde_per_year']:.1%} per year")
    print(f"Mean exposure:\t{mean_exposure:.1%}\tZero-skill reference at that exposure:\t{zero_skill_reference:.2%}")
    print(f"PRIMARY on the prior-only path (chained quarters above buy & hold):\t{prior_summary_dict['chained_return'] > prior_summary_dict['chained_buy_hold_return']}")
    # PREVIEW THE INFORMATION TEST
    display(baseline_summary_pdf.round(4))

Prior-only path (44 quarters):	146.94% vs buy & hold 218.48%	(beats buy & hold in 15 quarters, one-sided sign test p = 0.989)
Mean excess per quarter:	-0.56% (SD 3.38%, t = -1.11)	Minimum detectable effect:	1.43% per quarter, about 5.7% per year
Mean exposure:	88.1%	Zero-skill reference at that exposure:	177.41%
PRIMARY on the prior-only path (chained quarters above buy & hold):	False


,baseline,chained_return,model_beats_random_run_share,random_run_count,model_chained_return,model_beats
0,fixed_time,0.1887,NaN,NaN,1.4694,True
1,random_* (median of runs),1.2455,0.8,20.0,1.4694,True


In [10]:
# IF THE PRIOR-ONLY PATH HAS ROWS
if prior_row_dict_list:
    # CREATE A FIGURE OF THE PRIOR-ONLY PATH, BUY AND HOLD AND THE RANDOM MEDIAN (CHAINED QUARTERS)
    random_median_pdf = prior_baseline_pdf[prior_baseline_pdf["baseline"].str.startswith("random_")].groupby("fold_id")["total_return"].median()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=prior_only_pdf["fold_id"], y=np.cumprod(1 + prior_only_pdf["valid_total_return"]), mode="lines+markers", name="Model (prior-only)"))
    fig.add_trace(go.Scatter(x=prior_only_pdf["fold_id"], y=np.cumprod(1 + prior_only_pdf["valid_buy_hold_total_return"]), mode="lines+markers", name="Buy & Hold"))
    fig.add_trace(go.Scatter(x=random_median_pdf.index, y=np.cumprod(1 + random_median_pdf.to_numpy()), mode="lines+markers", name="Random entry (median per quarter)"))
    fig.update_layout(title="Prior-Only Path On Validation Quarters (growth of 1)", template="plotly_dark", width=1100, height=500, font=dict(color="white"))
    fig.show()
    # LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATES ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
    summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": "previous quarter", "path": "prior-only on validation"},
                                        {**prior_summary_dict, "mean_exposure": mean_exposure, "zero_skill_reference_return": zero_skill_reference,
                                         "baseline_chained_return_dict": {row["baseline"]: row["chained_return"] for _, row in baseline_summary_pdf.iterrows()},
                                         "best_after_the_fact_chained_return": float(candidate_summary_pdf["chained_return"].iloc[0])},
                                        False, note_str_in="exp01 step03 validation summary (prior-only path, baselines, exposure, MDE)")

In [11]:
# CONVERT THE TEST FOLD SUMMARIES TO A DATAFRAME
fold_summary_pdf = pd.DataFrame(fold_summary_dict_list)
# IF THERE ARE TEST RESULTS
if not fold_summary_pdf.empty:
    # SAVE THE FOLD SUMMARY
    write_csv_file_to_path(fold_summary_pdf, f"{output_path_str}fold_summary_data.csv", "W")
    # SUMMARIZE THE TEST WINDOWS
    test_summary_dict = get_window_return_summary_dict(fold_summary_pdf["total_return"].to_numpy(), fold_summary_pdf["buy_hold_return"].to_numpy())
    # DISPLAY INFORMATION
    print(f"Folds Beating Buy & Hold:\t{test_summary_dict['windows_beating_buy_hold']}/{test_summary_dict['window_count']}\t(one-sided sign test p-value: {test_summary_dict['sign_test_p_value']:.4f})")
    print(f"Chained Model Return:\t\t{test_summary_dict['chained_return']:.2%}")
    print(f"Chained Buy & Hold Return:\t{test_summary_dict['chained_buy_hold_return']:.2%}")
    print(f"Mean Sharpe (model / B&H):\t{fold_summary_pdf['sharpe_ratio'].mean():.2f} / {fold_summary_pdf['buy_hold_sharpe_ratio'].mean():.2f}")
    print(f"Worst Drawdown (model / B&H):\t{fold_summary_pdf['max_drawdown'].min():.2%} / {fold_summary_pdf['buy_hold_max_drawdown'].min():.2%}")
    print(f"Total Trades:\t\t\t{int(fold_summary_pdf['trade_count'].sum()):,}")
# PREVIEW DATAFRAME
fold_summary_pdf

""


In [12]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,1,1,0,2026-10-05T15:17:38,2026-10-05T15:17:38
1,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
2,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
3,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
4,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
5,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
6,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
7,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
8,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
9,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
